In [4]:
!pip install numpy tqdm

  Using cached tqdm-4.67.1-py3-none-any.whl.metadata (57 kB)
Using cached tqdm-4.67.1-py3-none-any.whl (78 kB)


In [27]:
# Path to the local clone of the dataset repo (must contain train.csv and test.csv)
DATA_REPO_DIR = "../sudoku-extreme"   # cloned sapientinc/sudoku-extreme

# Optional difficulty filter (as in builder)
MIN_DIFFICULTY = None  # 5; or None to skip filtering

# Sampling / augmentation (match original pipeline logic)
TRAIN_SUBSAMPLE = 1000     # subsample only TRAIN (None => use all train rows)
AUG_PER_SEED    = 1000      # number of augmentations per train seed
INCLUDE_ORIGINAL = True    # also check the original board as a candidate for leakage

# Reproducibility and diagnostics
RANDOM_SEED = 0            # np.random seed for reproducibility
CHECK_INPUT_AND_SOLUTION = False  # if True, match on (input,solution) pairs; else match input only
PRINT_EXAMPLES = 3         # print a few collision examples (0 => don't print)

In [22]:
import os, csv
import numpy as np
from tqdm import tqdm

# rng = np.random.default_rng(RANDOM_SEED)

def parse_board_0_9(q: str) -> np.ndarray:
    """
    Convert an 81-char string ('.' or '1'..'9') into a 9x9 array of uint8 in 0..9 (0=blank).
    """
    arr = np.frombuffer(q.replace('.', '0').encode(), dtype=np.uint8) - ord('0')
    return arr.reshape(9, 9).astype(np.uint8)

def load_csv_split(repo_dir: str, split: str, min_difficulty=None):
    """
    Read <split>.csv and return lists of (inputs_9x9, labels_9x9) in 0..9 space.
    CSV format: source, q, a, rating. Skip header; optionally filter by rating.
    """
    path = os.path.join(repo_dir, f"{split}.csv")
    inputs, labels = [], []
    with open(path, newline="", encoding="utf-8") as f:
        reader = csv.reader(f)
        _ = next(reader, None)  # skip header if present
        for row in reader:
            if len(row) != 4:
                continue
            source, q, a, rating = row
            if (min_difficulty is None) or (int(rating) >= int(min_difficulty)):
                q_arr = parse_board_0_9(q)
                a_arr = parse_board_0_9(a)
                inputs.append(q_arr)
                labels.append(a_arr)
    return inputs, labels

def board_bytes(b9: np.ndarray) -> bytes:
    """
    Flatten a 9x9 board and convert to bytes for exact-equality hashing.
    """
    return b9.reshape(-1).tobytes()

def pair_bytes(inp9: np.ndarray, sol9: np.ndarray) -> bytes:
    """
    Concatenate input and solution (flattened) and convert to bytes for exact-equality hashing of pairs.
    """
    return np.concatenate([inp9.reshape(-1), sol9.reshape(-1)]).tobytes()

def to_hashset_inputs(boards):
    """
    Hashset of inputs only (exact-equality on flattened 81 tokens).
    """
    return { board_bytes(b) for b in boards }

def to_hashset_pairs(inputs, labels):
    """
    Hashset of (input,solution) pairs (exact-equality).
    """
    return { pair_bytes(i, l) for i, l in zip(inputs, labels) }

In [23]:
def shuffle_sudoku(board: np.ndarray, solution: np.ndarray):
    """
    Apply the same family of Sudoku symmetries as the builder:
      - digit permutation on 1..9 (0 -> 0),
      - optional transpose,
      - permute bands/stacks (3x3) and rows/cols within them,
      - consistent 81->81 index mapping.
    Inputs/outputs are 9x9 in 0..9 space (0=blank).
    """
    # 1) digit permutation: map 1..9 randomly; preserve 0
    digit_map = np.pad(np.random.permutation(np.arange(1, 10)), (1, 0))

    # 2) optional transpose
    transpose_flag = np.random.random() < 0.5

    # 3) permute row bands and rows within each band
    bands = np.random.permutation(3)
    row_perm = np.concatenate([b * 3 + np.random.permutation(3) for b in bands])

    # 4) permute column stacks and columns within each stack
    stacks = np.random.permutation(3)
    col_perm = np.concatenate([s * 3 + np.random.permutation(3) for s in stacks])

    # 5) build mapping for flat indices 0..80
    mapping = np.array([row_perm[i // 9] * 9 + col_perm[i % 9] for i in range(81)])

    def apply(x: np.ndarray) -> np.ndarray:
        xx = x.T if transpose_flag else x
        new_board = xx.flatten()[mapping].reshape(9, 9).copy()
        # digit remap (digit_map[0] == 0 keeps blanks)
        return digit_map[new_board]

    return apply(board), apply(solution)

In [12]:
train_inputs, train_labels = load_csv_split(DATA_REPO_DIR, "train", MIN_DIFFICULTY)
test_inputs,  test_labels  = load_csv_split(DATA_REPO_DIR, "test",  MIN_DIFFICULTY)

print(f"Loaded from CSV: train={len(train_inputs):,}, test={len(test_inputs):,}")

# Subsample TRAIN only (matching original behavior)
if TRAIN_SUBSAMPLE is not None and TRAIN_SUBSAMPLE < len(train_inputs):
    idx = np.random.choice(len(train_inputs), size=TRAIN_SUBSAMPLE, replace=False)
    train_inputs = [train_inputs[i] for i in idx]
    train_labels = [train_labels[i] for i in idx]
    print(f"Train subsampled to {len(train_inputs):,}")

Loaded from CSV: train=3,831,994, test=422,786
Train subsampled to 1,000


In [13]:
if CHECK_INPUT_AND_SOLUTION:
    test_hash = to_hashset_pairs(test_inputs, test_labels)  
    print(f"Test hash (pairs): {len(test_hash):,}")
else:
    test_hash = to_hashset_inputs(test_inputs)             
    print(f"Test hash (inputs): {len(test_hash):,}")

Test hash (inputs): 422,786


In [28]:
total_aug = 0
total_collisions = 0
seeds_with_collision = 0
examples = []   # (seed_index, "orig"/"aug", (inp, sol)) for a few prints

for si, (inp9, sol9) in enumerate(tqdm(list(zip(train_inputs, train_labels)), desc="Scanning seeds")):
    had = False

    # Check original (optional)
    if INCLUDE_ORIGINAL:
        key = pair_bytes(inp9, sol9) if CHECK_INPUT_AND_SOLUTION else board_bytes(inp9)
        if key in test_hash:
            total_collisions += 1
            had = True
            if len(examples) < PRINT_EXAMPLES:
                examples.append((si, "orig", (inp9.copy(), sol9.copy())))
        total_aug += 1

    # Augmentations
    for _ in range(AUG_PER_SEED):
        aug_inp, aug_sol = shuffle_sudoku(inp9, sol9)
        key = pair_bytes(aug_inp, aug_sol) if CHECK_INPUT_AND_SOLUTION else board_bytes(aug_inp)
        if key in test_hash:
            total_collisions += 1
            had = True
            if len(examples) < PRINT_EXAMPLES:
                examples.append((si, "aug", (aug_inp.copy(), aug_sol.copy())))
        total_aug += 1

    if had:
        seeds_with_collision += 1

print("\n=== Leakage stats (train subsample + aug vs FULL test) ===")
print(f"Train seeds scanned          : {len(train_inputs):,}")
print(f"Augmentations per seed       : {AUG_PER_SEED} (+1 original: {INCLUDE_ORIGINAL})")
print(f"Total aug+orig generated     : {total_aug:,}")
print(f"Test boards indexed          : {len(test_inputs):,} (FULL)")
print(f"Collisions (exact matches)   : {total_collisions:,}")
rate = (total_collisions / max(total_aug, 1)) * 100.0
print(f"Collision rate               : {rate:.6f}%")
print(f"Seeds with ≥1 collision      : {seeds_with_collision:,} / {len(train_inputs):,} "
      f"({(seeds_with_collision / max(len(train_inputs),1))*100:.2f}%)")

Scanning seeds: 100%|██████████| 1000/1000 [00:36<00:00, 27.62it/s]


=== Leakage stats (train subsample + aug vs FULL test) ===
Train seeds scanned          : 1,000
Augmentations per seed       : 1000 (+1 original: True)
Total aug+orig generated     : 1,001,000
Test boards indexed          : 422,786 (FULL)
Collisions (exact matches)   : 0
Collision rate               : 0.000000%
Seeds with ≥1 collision      : 0 / 1,000 (0.00%)
